# attempt25_f1.ipynb — Fold 1 ResNeXt50 inference

Inference-only notebook. Loads the best fold-1 checkpoint from attempt_25
(`best_resnext50_fold0_s2.pth` if Stage 2 improved, else `best_resnext50_fold0_s1.pth`)
and generates a Codabench submission for the FINE-TUNING category.

- Architecture : ResNeXt50-32x4d (IMAGENET1K_V2), head = Dropout(0.5) → Linear(2048, 1)
- Resolution   : 380 px (CropByEye → BenGraham → Rescale(416) → CenterCrop(380))
- TTA          : 6-pass (orig + hflip + vflip + rot180 + rot90 + rot270)

In [ ]:
from __future__ import print_function, division
import os, csv
import torch
import pandas as pd
from skimage import io, transform, util, color
from sklearn import metrics
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnext50_32x4d, ResNeXt50_32X4D_Weights
import torchvision.transforms.functional as TF
import torch.nn as nn
from PIL import Image
from zipfile import ZipFile
import random
import numpy.random as npr
import cv2
import warnings

warnings.filterwarnings('ignore')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
DATA_ROOT  = '/kaggle/input/datasets/mariamuozperez/lab5-cv'
SAVE_PATH  = '/kaggle/working'

# Prefer Stage 2 checkpoint; fall back to Stage 1
s2_path = os.path.join(SAVE_PATH, 'best_resnext50_fold0_s2.pth')
s1_path = os.path.join(SAVE_PATH, 'best_resnext50_fold0_s1.pth')
MODEL_PATH = s2_path if os.path.exists(s2_path) else s1_path
print(f'Loading: {MODEL_PATH}')

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
class RetinopathyDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.dataset  = pd.read_csv(csv_file, header=0,
                                    dtype={'id': str, 'eye': int, 'label': int})
        self.root_dir = root_dir
        self.img_dir  = os.path.join(root_dir, 'images')
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset.iloc[idx]
        img_name = os.path.join(self.img_dir, row['id'] + '.jpg')
        image = io.imread(img_name)
        if row['eye'] == 1:
            image = image[:, ::-1, :]
        sample = {
            'image': image,
            'eye':   int(row['eye']),
            'label': int(row['label'] > 0),
        }
        if self.transform:
            sample = self.transform(sample)
        return sample


# ── Transforms ───────────────────────────────────────────────────────────────
class CropByEye(object):
    def __init__(self, threshold=0.10, border=1):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        imgray = color.rgb2gray(image)
        _, mask = cv2.threshold(imgray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return {'image': image, 'eye': eye, 'label': label}
        minx = np.maximum(sidx[1].min() - self.border[1], 0)
        maxx = np.minimum(sidx[1].max() + 1 + self.border[1], w)
        miny = np.maximum(sidx[0].min() - self.border[0], 0)
        maxy = np.minimum(sidx[0].max() + 1 + self.border[1], h)
        return {'image': image[miny:maxy, minx:maxx, ...], 'eye': eye, 'label': label}


class BenGraham(object):
    def __init__(self, sigmaX=10):
        self.sigmaX = sigmaX

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        img_u8 = image if image.dtype == np.uint8 else (np.clip(image, 0, 1) * 255).astype(np.uint8)
        blurred  = cv2.GaussianBlur(img_u8, (0, 0), self.sigmaX)
        enhanced = cv2.addWeighted(img_u8, 4, blurred, -4, 128)
        enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
        mask = np.zeros(enhanced.shape, dtype=np.uint8)
        h, w = enhanced.shape[:2]
        cv2.circle(mask, (w // 2, h // 2), int(0.9 * min(h, w) / 2), (1, 1, 1), -1, 8, 0)
        enhanced = enhanced * mask + 128 * (1 - mask)
        return {'image': enhanced.astype(np.float32) / 255.0, 'eye': eye, 'label': label}


class Rescale(object):
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            new_h = self.output_size * h / w if h > w else self.output_size
            new_w = self.output_size if h > w else self.output_size * w / h
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (int(new_h), int(new_w)))
        return {'image': image, 'eye': eye, 'label': label}


class CenterCrop(object):
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = int((h - new_h) / 2) if h > new_h else 0
        left = int((w - new_w) / 2) if w > new_w else 0
        return {'image': image[top:top+new_h, left:left+new_w], 'eye': eye, 'label': label}


class ToTensor(object):
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        image = torch.from_numpy(image.transpose((2, 0, 1)))
        return {'image': image, 'eye': eye, 'label': torch.tensor(label, dtype=torch.long)}


class Normalize(object):
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        mean = torch.as_tensor(self.mean, dtype=image.dtype, device=image.device)
        std  = torch.as_tensor(self.std,  dtype=image.dtype, device=image.device)
        image.sub_(mean[:, None, None]).div_(std[:, None, None])
        return {'image': image, 'eye': eye, 'label': label}

In [ ]:
# ── Eval pipeline (380 px) ───────────────────────────────────────────────────
pixel_mean = [0.485, 0.456, 0.406]
pixel_std  = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(416),
    CenterCrop(380),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

val_dataset  = RetinopathyDataset(os.path.join(DATA_ROOT, 'val.csv'),  DATA_ROOT, transform=eval_transform)
test_dataset = RetinopathyDataset(os.path.join(DATA_ROOT, 'test.csv'), DATA_ROOT, transform=eval_transform)

val_loader  = DataLoader(val_dataset,  batch_size=128, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

print(f'Val: {len(val_dataset)}  Test: {len(test_dataset)}')

In [ ]:
# ── Load model ───────────────────────────────────────────────────────────────
model = resnext50_32x4d(weights=None)
model.fc = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(model.fc.in_features, 1))
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()
print(f'Loaded: {MODEL_PATH}')
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 6-pass TTA inference ─────────────────────────────────────────────────────
def score_loader(model, loader, tta=True):
    model.eval()
    n = len(loader.dataset)
    scores = np.zeros((n, 1), dtype=np.float32)
    labels = np.zeros((n,),   dtype=np.int64)
    cont = 0
    with torch.no_grad():
        for sample in loader:
            x  = sample['image'].to(device).float()
            bs = x.shape[0]
            if tta:
                s1 = torch.sigmoid(model(x))
                s2 = torch.sigmoid(model(torch.flip(x, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(x, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(x, dims=[2, 3])))
                s5 = torch.sigmoid(model(torch.rot90(x, 1, [2, 3])))
                s6 = torch.sigmoid(model(torch.rot90(x, 3, [2, 3])))
                out = (s1 + s2 + s3 + s4 + s5 + s6) / 6.0
            else:
                out = torch.sigmoid(model(x))
            scores[cont:cont+bs] = out.cpu().numpy()
            labels[cont:cont+bs] = sample['label'].numpy()
            cont += bs
    return scores, labels

In [ ]:
# ── Sanity check: val AUC ────────────────────────────────────────────────────
val_scores, val_labels = score_loader(model, val_loader, tta=True)
val_auc = metrics.roc_auc_score(val_labels, val_scores)
print(f'Val AUC (TTA-6, original val.csv): {val_auc:.4f}')

In [ ]:
# ── Test inference ───────────────────────────────────────────────────────────
test_scores, _ = score_loader(model, test_loader, tta=True)
print(f'Test scores shape: {test_scores.shape}')  # expected (1000, 1)

In [ ]:
# ── Save submission ──────────────────────────────────────────────────────────
out_ft_path  = os.path.join(SAVE_PATH, 'output_ft.csv')
zip_path     = os.path.join(SAVE_PATH, 'codabench_submission.zip')

with open(out_ft_path, mode='w', newline='') as f:
    csv.writer(f).writerows(test_scores)

with ZipFile(zip_path, 'w') as zf:
    zf.write(out_ft_path, arcname='output_ft.csv')

print(f'Saved: {out_ft_path}')
print(f'Saved: {zip_path}')
print(f'Val AUC (TTA-6): {val_auc:.4f}')